# Commodity Option Strategy Advisor — Interactive Panel (ipywidgets)

Widget-driven version of `option_strategy_advisor.ipynb`. All pricing logic
(Black-76 engine, strategy library, decision matrix) lives in `advisor_core.py`,
shared with the Streamlit version (`streamlit_app.py`).

**Use it two ways:**
1. **In Jupyter** — run all cells, then drive the dropdowns below.
2. **As a web app** — serve it with Voila (code cells hidden, widgets live):
   ```bash
   voila option_strategy_advisor_widgets.ipynb
   ```

Pick your commodity, role, price view, and premium appetite — the recommendation,
trade card, and payoff diagram update automatically. Futures price and IV default
to the commodity profile but can be overridden with live levels.

In [ ]:
%matplotlib inline
import ipywidgets as widgets
import matplotlib.pyplot as plt
from IPython.display import display, clear_output

from advisor_core import (
    COMMODITIES, STRATEGIES, ROLES, VIEWS, PREMIUM_APPETITES, TENORS,
    analyze_strategy, recommend, payoff_figure,
)


def print_trade_card(res):
    """Text trade card, same format as the original notebook."""
    W = 72
    print('=' * W)
    print(f"  {res['strategy'].upper()}  on  {res['commodity']}")
    print('=' * W)
    print(f"  Market: F={res['F']} {res['unit']} | IV={res['sigma'] * 100:.0f}% | "
          f"T={res['T'] * 12:.0f} months | {res['exchange']}")
    print(f"  {res['summary']}")
    print('-' * W)
    print(f"  {'Side':<6}{'Qty':>4}  {'Type':<6}{'Strike':>10}  {'Premium':>10}")
    for leg in res['legs']:
        p_str = f"{leg['premium']:>10.4f}" if leg['premium'] is not None else f"{'—':>10}"
        print(f"  {leg['side']:<6}{leg['qty']:>4}  {leg['type']:<6}{leg['strike']:>10.2f}  {p_str}")
    print('-' * W)
    net = res['net_premium']
    net_lbl = 'you PAY' if net > 0 else 'you COLLECT'
    print(f"  Net premium: {abs(net):.4f} {res['unit']}  ({net_lbl})")
    print(f"  Per contract ({res['contract_size']:,} {res['contract_unit']}): "
          f"${abs(net) * res['contract_size']:,.0f} {net_lbl.split()[-1].lower()}")
    if res['breakevens']:
        print(f"  Breakeven(s) at expiry: {', '.join(str(b) for b in res['breakevens'])} {res['unit']}")
    print(f"  Max profit: {res['max_profit']:,.2f} {res['unit']}/unit   "
          f"Max loss: {res['max_loss']:,.2f} {res['unit']}/unit  (within ±50% price range shown)")
    print('-' * W)
    g = res['greeks']
    print(f"  Net Greeks (per unit): D={g['delta']:+.3f}  G={g['gamma']:+.4f}  "
          f"Th={g['theta']:+.4f}/day  Vega={g['vega']:+.4f}/1%IV")
    print(f"  When to use: {res['when']}")
    print(f"  Watch out: {res['watch']}")
    print(f"  Commodity note: {res['commodity_notes']}")
    print('=' * W)


w_commodity = widgets.Dropdown(options=list(COMMODITIES), value='WTI Crude', description='Commodity:')
w_role = widgets.Dropdown(options=ROLES, value='Producer', description='Your role:')
w_view = widgets.Dropdown(options=VIEWS, value='Bearish / fear a fall', description='Price view:')
w_prem = widgets.Dropdown(options=PREMIUM_APPETITES, value='Pay premium', description='Premium:')
w_tenor = widgets.Dropdown(options=TENORS, value=0.25, description='Tenor:')
w_F = widgets.BoundedFloatText(value=COMMODITIES['WTI Crude']['F'], min=0.01, max=1e6,
                               step=0.5, description='Futures F:')
w_iv = widgets.IntSlider(value=int(COMMODITIES['WTI Crude']['sigma'] * 100), min=5, max=150,
                         description='IV %:', continuous_update=False)
w_strat = widgets.Dropdown(options=['(recommended)'], value='(recommended)', description='Analyze:')

out = widgets.Output()
_syncing = False


def refresh(_=None):
    if _syncing:
        return
    with out:
        clear_output(wait=True)
        picks = recommend(w_role.value, w_view.value, w_prem.value)
        if picks is None:
            print('No mapping for these inputs — check your selections.')
            return
        primary, alternatives = picks
        # keep the "Analyze" dropdown in sync with the current recommendation
        opts = ['(recommended)'] + list(alternatives)
        if list(w_strat.options) != opts:
            w_strat.unobserve(refresh, names='value')
            w_strat.options = opts
            w_strat.value = '(recommended)'
            w_strat.observe(refresh, names='value')
        chosen = primary if w_strat.value == '(recommended)' else w_strat.value

        print(f"YOUR SITUATION:  {w_role.value} | {w_view.value} | {w_prem.value} | {w_commodity.value}")
        print(f"RECOMMENDED:     > {primary}")
        if alternatives:
            print(f"ALTERNATIVE(S):    {', '.join(alternatives)}  (pick one in the 'Analyze' dropdown to compare)")
        if chosen != primary:
            print(f"SHOWING:         {chosen}")
        print()
        res = analyze_strategy(chosen, w_commodity.value, T=w_tenor.value,
                               F=w_F.value, sigma=w_iv.value / 100)
        print_trade_card(res)
        fig = payoff_figure(res)
        display(fig)
        plt.close(fig)


def sync_market_defaults(_=None):
    """When the commodity changes, reset F and IV to its profile defaults."""
    global _syncing
    c = COMMODITIES[w_commodity.value]
    _syncing = True
    w_F.value = c['F']
    w_iv.value = int(c['sigma'] * 100)
    _syncing = False
    refresh()


w_commodity.observe(sync_market_defaults, names='value')
for w in (w_role, w_view, w_prem, w_tenor, w_F, w_iv, w_strat):
    w.observe(refresh, names='value')

display(widgets.VBox([
    widgets.HBox([w_commodity, w_role, w_view]),
    widgets.HBox([w_prem, w_tenor, w_strat]),
    widgets.HBox([w_F, w_iv]),
    out,
]))
refresh()

---
> **Disclaimer:** default prices and vols in `advisor_core.COMMODITIES` are illustrative.
> Update them to live market levels, and treat the output as a structured starting point — not trade advice.